# Performer
You can __[download](https://arxiv.org/pdf/2009.14794)__ and read the paper. Also, there is a __[video](https://www.youtube.com/watch?v=xJrKIPwVwGM&t=76s)__ describes paper.

Here we want to implement Performer on __[LLAMA3](https://huggingface.co/meta-llama/Llama-3.2-1B)__ and compare it with Vanilla Transformer Attention mechanism.
## What expect?
Vanilla attention is O(L^2) and performer is O(L). There for, Performer should be faster and need less time and memmory.
## A brief look at the formulas
Vanilla transformer mechanism uses formula below to calculate:
$$\text{Attention}(Q, K, V) = \text{softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right) V$$
For Performer attention we use (FAVOR+):
$$\text{Attention}_{\text{FAVOR}+}(Q, K, V) = \frac{\Phi(Q) \left( \Phi(K)^\top V \right)}{\Phi(Q) \left( \Phi(K)^\top \mathbf{1} \right)}$$
## Implement Libraries that you need
Here we implemented Performer attention.

NOTE: Even we implemented vanilla attention, you can load LLAMA3 with vanilla attention. Just use line below:
```python
attn_implementation="eager"
```
### You need to run Language model downloaded from hugging face?
If You need some language model from hugging face, you need to implement huggingface_hub and use login function.

Use code below to implement it:
```python
from huggingface_hub import login
login()
```

You need __[Access Token](https://huggingface.co/docs/hub/en/security-tokens)__ for that. You can __[crete your access token](https://huggingface.co/settings/tokens)__ by your own.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import torch.nn as nn
import math
import bitsandbytes as bnb
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
import gc
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb
from huggingface_hub import login
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.notebook import tqdm
import json

# Do you want to use huggingface models?
First login to huggingface

In [ ]:
login()

## Load your model
You can load any model that you want. Just use huggingface link pass it as funtion input. As default model, we set it LLAMAA-3.2-1B and quantized it to 4bit.
### Want to use small models?
Take it easy. Just disable quantization using quantize parameter. Make it false
### SMALL TIP
If you use small models, you may not get your prefered output. Use models that can handel inputs with more than 4096 inputs (The reason is mentioned in next cells).

In [ ]:
def generate_model(model_id="meta-llama/Llama-3.2-1B", quantize=True, device="cuda", with_tokenizer=True):

    if quantize:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    else:
        bnb_config = None
        
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    ).to(device)
    model = prepare_model_for_kbit_training(model)
    if with_tokenizer:
        return model, tokenizer
    return model

model, tokenizer = generate_model()

# Implement Performer function using this class
First we need to implement the matrixes with the sizes. As we use 4-bit quantized model, here we need to implement for this model too. So, there is an opthion `quantized` that you can set it True to implement.

$$\phi(x) = \frac{h(x)}{\sqrt{m}} \exp(W \cdot x)$$ $$h(x) = \exp\left(-\frac{\|x\|^2}{2}\right)$$

## Calculate output
To calculate output
### Implementation Logic (Causal FAVOR+)
The code implements the causal mechanism by separating the numerator (values) and the denominator (normalization) to achieve linear complexity $O(L \cdot d \cdot m)$:

$$\text{Output}_i = \frac{\phi(q_i)^\top \sum_{j=1}^i (\phi(k_j) \otimes v_j)}{\phi(q_i)^\top \sum_{j=1}^i \phi(k_j) + \epsilon}$$

#### Correspondence with Code:
* **Numerator:** `(q_prime * torch.cumsum(k_prime @ v, dim=2)).sum(dim=3)`
* **Denominator:** `(q_prime * torch.cumsum(k_prime, dim=2)).sum(dim=3)`
* **$\epsilon$:** `1e-6` (added for numerical stability).

Insted of using the code below, I used `torch.cumsum` to make the code faster:
```python
# فرض می‌کنیم ابعاد رو داری: b, h, t, m, d
kv_acc = torch.zeros((b, h, t, m, d), device=k_prime.device)
k_acc = torch.zeros((b, h, t, m), device=k_prime.device)

# تعریف متغیرهایی برای نگه داشتن جمع لحظه‌ای (Running Sum)
current_kv = 0
current_k = 0

for i in range(t):
    # جمع زدن مقادیر تا لحظه i
    current_kv = current_kv + k_v_outer[:, :, i]
    current_k = current_k + k_prime[:, :, i]
    
    # ذخیره در ماتریس‌های نهایی
    kv_acc[:, :, i] = current_kv
    k_acc[:, :, i] = current_k

# حالا محاسبات نهایی با استفاده از ضرب معمولی و مجموع (به جای einsum)
out_num = (q_prime.view(b, h, t, m, 1) * kv_acc).sum(dim=3)
out_den = (q_prime * k_acc).sum(dim=3)
```

In [ ]:
class PerformerSelfAttention(nn.Module):
    def __init__(self, config, nb_features=256, quantized=True):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = self.hidden_size // self.num_heads
        self.num_groups = self.num_heads // self.num_kv_heads
        self.nb_features = nb_features

        if quantized:
            # 4-bit quantized parameter set
            self.q_proj = bnb.nn.Linear4bit(
                self.hidden_size, self.num_heads * self.head_dim, bias=False,
                compute_dtype=torch.bfloat16,
                quant_type="nf4"
            )
            self.k_proj = bnb.nn.Linear4bit(
                self.hidden_size, self.num_kv_heads * self.head_dim, bias=False,
                compute_dtype=torch.bfloat16,
                quant_type="nf4"
            )
            self.v_proj = bnb.nn.Linear4bit(
                self.hidden_size, self.num_kv_heads * self.head_dim, bias=False,
                compute_dtype=torch.bfloat16,
                quant_type="nf4"
            )
            self.o_proj = bnb.nn.Linear4bit(
                self.num_heads * self.head_dim, self.hidden_size, bias=False,
                compute_dtype=torch.bfloat16,
                quant_type="nf4"
            )
        else:
            # Standard parameter set
            self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
            self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
            self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
            self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)

        projection_matrix = torch.randn(self.nb_features, self.head_dim)
        q, _ = torch.qr(projection_matrix.T) # Orthogonalize for better accuracy
        self.register_buffer("projection_matrix", q.T)

    def repeat_kv(self, hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
        batch, num_key_value_heads, slen, head_dim = hidden_states.shape
        if n_rep == 1:
            return hidden_states
        hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
        return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

    def feature_map(self, x):
        # Project x onto the random features
        x_proj = x @ self.projection_matrix.T

        # Calculate norm term for the softmax approximation: ||x||^2 / 2
        x_norm = torch.sum(x**2, dim=-1, keepdim=True) / 2.0

        # The positive feature map (FAVOR+ SMU kernel)
        phi = torch.exp(x_proj - x_norm - x_proj.max(dim=-1, keepdim=True)[0]) # Max-subtraction for stability
        return phi / math.sqrt(self.nb_features)

    def forward(self, hidden_states, attention_mask=None, position_ids=None, **kwargs):
        Batch, T, C = hidden_states.size()

        q = self.q_proj(hidden_states).view(Batch, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(Batch, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(Batch, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        k = self.repeat_kv(k, self.num_groups)
        v = self.repeat_kv(v, self.num_groups)

        q_prime = self.feature_map(q) # (B, H, T, M)
        k_prime = self.feature_map(k) # (B, H, T, M)

        batch, h, t, m = k_prime.shape # (B, H, T, M, D)
        d = v.shape[-1]

        k_v_outer = k_prime.view(batch, h, t, m, 1) * v.view(batch, h, t, 1, d)
        batch, h, t, m = q_prime.shape

        kv_cumsum = torch.cumsum(k_v_outer, dim=2)
        out_num = (q_prime.view(batch, h, t, m, 1) * kv_cumsum).sum(dim=3)
        k_cumsum = torch.cumsum(k_prime, dim=2) # (B, H, T, M)
        out_den = (q_prime * k_cumsum).sum(dim=3)
        out_den = out_den.unsqueeze(-1) + 1e-6 # Add epsilon for stability

        # 4. Normalize and Project
        output = out_num / out_den
        output = output.transpose(1, 2).contiguous().view(Batch, T, C)

        return self.o_proj(output), None

# Changing attention heads
for changing attention heads in LLAMA3.2, you should use ```(your model name).model.layers``` to get access to each layer and the ```self_attn``` in each layer shows the attention mechanism of the attention head. Therefor, you should change this parameter.

## DO NOT FORGET
It is really important to run this head at same place with hole model

In [ ]:
for layer in model.model.layers:
    layer.self_attn = PerformerSelfAttention(layer.self_attn.config).to("cuda")

# Efficient Training Setup (LoRA)
This configuration allows training large models with minimal hardware (VRAM) and high efficiency
## LoRA (Low-Rank Adaptation)
Instead of updating billions of parameters, it trains only small "adapter" layers ($r=32$), saving memory and time.

In [ ]:
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.gradient_checkpointing_enable()

# You have saved checkepoints?
Use method this method to load your model using checkpoints

In [ ]:
model = model.from_pretrained(model, "/content/PerformerSelfAttention/content/PerformerSelfAttention/checkpoint-200")

# Fine-tuning step
Use step below to fine-tune your model.
## IMPORTANT
Take care of your quantization. step below is for 4-bit quantized model and if you use it for unquantized model, it will underfit. Also, using unquantized parameters for 4-bit quantized, causes overfitting.

### Parameters meaning
```per_device_train_batch_size``` is the number of batchs going in GPU. If your GPU is powerful enough, take it easy, increase it.

```gradient_accumulation_steps``` creates batch size and calculates then update.

By timing two previous parameters, your real batch size will be showen.

In [ ]:
print("\n--- Starting Fine-tuning ---")
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./PerformerSelfAttention",
    optim="paged_adamw_32bit",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_steps=200,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.3,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    gradient_checkpointing=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer.train()

print("Fine-tuning complete.")

# Create long sequences
You need to create long datasets to evaluate model with long sequences.

1640 data is enough for evaluating. even with this number, it may takes hours.

In [ ]:
def create_data_set(dataset, num_samples, sequence_length=4100):
    chunks = []
    current_text = ""

    for i in tqdm(range(min(num_samples, len(dataset)))):
        if (len(chunks) == 1640) and ((sequence_length is not None)):
            break
        text = dataset[i]["text"]

        if not text.strip():
            continue

        candidate_text = current_text + text

        tokens = tokenizer(
            candidate_text,
            return_tensors="pt",
            truncation=False
        )

        if (sequence_length is not None) and (tokens["input_ids"].shape[1] <= sequence_length):
            current_text = candidate_text
        else:
            if current_text:
                chunks.append(
                    tokenizer(
                        current_text,
                        return_tensors="pt",
                        truncation=True,
                        max_length=sequence_length+100
                    )
                )
            current_text = text

    if current_text:
        chunks.append(
            tokenizer(
                current_text,
                return_tensors="pt",
                truncation=True,
                max_length=sequence_length+100
            )
        )
    return chunks

# Load enemy :)
Load LLAMA with its original attention model

In [ ]:
original_model = generate_model(with_tokenizer=False)

# Check how you fine-tuned
Use simple forward-pass for original model on fine-tuning dataset. Use the result to check what did you do in fine-tuning

In [ ]:
test_data = dataset["train"]
test_data = create_data_set(test_data, 100000, None)

total_loss = 0

with torch.no_grad():
    for i in tqdm(range(len(test_data))):

        inputs = test_data[i].to("cuda")
        outputs = original_model(**inputs, labels=inputs["input_ids"])
        oloss = outputs.loss

        if oloss is not None:
            current_loss = oloss.item()
        total_loss += current_loss

original_loss = total_loss / len(test_data)
print(f"Final Average Loss: {original_loss:.4f}")

# Plot the losses
Use step below to plot the fine-tuning loss and the loss of original data on dataset

In [ ]:
df = pd.DataFrame(trainer.state.log_history)

train_data = df[df['loss'].notna()]
eval_data = df[df['eval_loss'].notna()]

plt.figure(figsize=(10, 6))
ol = [original_loss for i in range(len(train_data['loss']))]
plt.plot(train_data['step'], train_data['loss'],
         label="Training Loss (Raw)", alpha=0.2, color='blue')
plt.plot(train_data['step'], train_data['loss'].rolling(window=5).mean(),
         label="Training Loss (Smoothed)", color='blue', linewidth=2)
plt.plot(train_data['step'], ol,
         label="Original LLAMA Loss", color='green', linewidth=2)

plt.plot(eval_data['step'], eval_data['eval_loss'],
         label="Validation Loss", color='red', marker='o', linestyle='--')

plt.title("Llama-3.2-1B Performer Fine-Tuning: Loss vs. Steps", fontsize=14)
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("Cross-Entropy Loss", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.savefig("fine_tuning_results.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Final Training Loss: {train_data['loss'].iloc[-1]:.4f}")
print(f"Final Validation Loss: {eval_data['eval_loss'].iloc[-1]:.4f}")

# Evaluate model using function below
here you can eveluate each model you want with any dataset. `starter` and `ender` are parameters to save the time to evaluate time latency. Use `output.loss.item()` function to evaluate loss. To mesure accuracy, you need to delete first lables (they are not predicted by model from logits) and delete last logits (we don't have the currect lable for them). For that you can use `logits[..., :-1, :]` and `labels[..., 1:]`. Get predictions using argmax and for getting the accuracy, run code below:
``` python
preds_list = predictions.flatten().tolist()
labels_list = shift_labels.flatten().tolist()

correct_count = 0
for p, l in zip(preds_list, labels_list):
    if l != -100:
        if p == l:
            correct_count += 1
```
But that may cause too long time. Use `correct = (predictions == shift_labels) & mask` insted

In [ ]:
def evaluate_and_measure(model, test_dataset, num_samples=50):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_tokens = 0
    latencies = []

    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)

    print(f"Evaluating {min(num_samples, len(test_dataset))} samples...")

    with torch.no_grad():
        for i in tqdm(range(min(num_samples, len(test_dataset)))):
            inputs = test_dataset[i].to("cuda")
            labels = inputs["input_ids"]

            starter.record()
            outputs = model(**inputs, labels=labels)
            ender.record()

            torch.cuda.synchronize()
            latencies.append(starter.elapsed_time(ender))

            total_loss += outputs.loss.item()

            logits = outputs.logits

            shift_logits = logits[..., :-1, :]
            shift_labels = labels[..., 1:]

            predictions = torch.argmax(shift_logits, dim=-1)

            mask = shift_labels != -100
            correct = (predictions == shift_labels) & mask

            total_correct += correct.sum().item()
            total_tokens += mask.sum().item()

    avg_loss = total_loss / min(num_samples, len(test_dataset))
    perplexity = np.exp(avg_loss)
    avg_latency = np.mean(latencies)
    accuracy = (total_correct / total_tokens) * 100 if total_tokens > 0 else 0

    return {
        "Perplexity": perplexity,
        "Accuracy (%)": accuracy,
        "Avg Latency (ms)": avg_latency,
        "Memory Allocated (GB)": torch.cuda.memory_allocated() / 1e9
    }

# Create dataset to Evaluate model
Here, you can implement dataset for evaluation. Thats much better to use other dataset rather than the dataset you use for fine-tuning. Load long enough dataset, something around 100,000 data.At last you can get the avrage sequence length of your dataset.

In [ ]:
dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:100000]")

print("creating dataset")
dataset = create_data_set(dataset, 10000000, 3000)
sum = 0
for i in range(len(dataset)):
    sum += len(dataset[i][0])
avg = sum // len(dataset)
print(f"Average sequence length is {avg}")
print(f"Loaded {len(dataset)} samples for evaluation.")

# Model evaluation
Use code below to evaluate model using previous function.

In [ ]:
print("\n--- PERFORMANCE OF PERFORMER-LLAMA ---")
model = model.to("cuda")
performer_results = evaluate_and_measure(model, dataset, 1640)
print(performer_results)

# Delete your model
Use lines below to delete your model from ram

### YOU CAN SKIP HERE IF YOU HAVE ENOUGH RAM

In [ ]:

del model
torch.cuda.empty_cache()
gc.collect()

# Save the results

In [ ]:
open(f"performer result of {avg} sequence length.json", "w").write(json.dumps(performer_results))

# Load baseline model
You can load baseline model to evaluate it

### Load it only if you didn't load befor

In [ ]:
print("\nLoading Original Baseline...")
original_model = generate_model(with_tokenizer=False)

In [ ]:
print("\n--- PERFORMANCE OF ORIGINAL LLAMA ---")
original_results = evaluate_and_measure(original_model, tokenizer, dataset, 1640)
print(original_results)

In [ ]:
open(f"performer result of {avg} sequence length.json", "w").write(json.dumps(performer_results))